# Phase 3: Skills Analysis and Temporal Trends

## Overview

This notebook implements Phase 3 of the mechanical engineering job analysis pipeline, combining the functionality of the original Phase 3 Parts 1 and 2. It analyzes skill requirements in mechanical engineering job postings over time (2010-2022) using advanced statistical modeling.

## Methodology

### Part 1: Skills Extraction and Prevalence Calculation
1. Parse BGT skill clusters from pipe-delimited format
2. Transform wide format (1 row/job) to long format (1 row/skill-job pair)
3. Calculate skill prevalence (proportion of jobs) by year
4. Filter to common skills (≥2% average prevalence)

### Part 2: Temporal Trend Analysis
1. Fit 5 candidate models (null, linear, log, exponential, quadratic)
2. Select best model using AICc with parsimony principle
3. Run diagnostic tests (Shapiro-Wilk, Breusch-Pagan, Durbin-Watson)
4. Assign two-dimensional evidence quality per B&A (2002):
   - **Discriminability**: how clearly we can pick a model winner (Strong / Moderate / Weak / Indistinguishable)
   - **Assumption Quality**: how much we trust the AICc values (Clean / Corrected / Compromised)
5. Classify trajectories (stable, growth, decline, accelerating, etc.); skills with Indistinguishable discriminability have no reported trajectory
6. Generate IEEE-format publication tables
7. Create trajectory and diagnostic visualizations

## Expected Outputs

- `skills_ME_ONLY.parquet`: ~9M skill-job pairs
- `skill_trajectory_summary.csv`: ~200-250 skill trajectories
- `regression_diagnostics.csv`: Diagnostic test results
- 8 IEEE-format tables (CSV + LaTeX)
- Trajectory plots by discriminability level
- Residual diagnostic plots per skill

## Setup and Configuration

In [1]:
# Import required libraries
import polars as pl
import numpy as np
import pandas as pd
import json
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Import utility modules
from utils import skills_utils, statistics_utils, visualization_utils, publication_utils

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Imports successful")
print(f"Working directory: {project_root}")

Imports successful
Working directory: /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022


In [2]:
# Load configuration
config_path = project_root / 'config' / 'analysis_config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

phase3_config = config['phase_3']
phase1_config = config['phase_1']  # Need base path

print("Phase 3 Configuration:")
print(f"  Input file: {phase3_config['input_file']}")
print(f"  Min prevalence: {phase3_config['min_prevalence']*100}%")
print(f"  Significance level (α): {phase3_config['alpha']}")
print(f"  Delta_i threshold: {phase3_config['delta_i_threshold']}")

Phase 3 Configuration:
  Input file: classified_ME_ONLY.parquet
  Min prevalence: 2.0%
  Significance level (α): 0.1
  Delta_i threshold: 2.0


# PART 1: Skills Extraction and Prevalence

## Load Classified Data

We load the NLP-classified jobs from Phase 2 and filter to "major" mechanical engineering positions. This ensures we're analyzing the core mechanical engineering workforce, not technicians or support roles.

In [3]:
# Check if cached skills file exists
skills_output_path = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['skills_output_file']

In [4]:
# Load data if not cached skills
if not skills_output_path.exists():
    print("=" * 80)
    print("PART 1: SKILLS EXTRACTION")
    print("=" * 80)

    # Load Phase 2 output
    input_path = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['input_file']

    print(f"\nLoading classified data from: {input_path}")
    study_data = pl.read_parquet(input_path)
    print(f"Loaded {len(study_data):,} records")

    # Filter to "major" mechanical engineering jobs
    major_data = study_data.filter(
        pl.col("NLP_Classification_Improvement_2").str.contains("major") & 
        ~pl.col("NLP_Classification_Improvement_2").str.contains("non-")
    )

    print(f"\nFiltered to {len(major_data):,} mechanical-major jobs")
    print(f"  Retention: {len(major_data)/len(study_data)*100:.1f}%")

    # Add Year and JobTextLength columns (before transformation for efficiency)
    major_data = major_data.with_columns([
        pl.col("JobDate").str.strptime(pl.Date, "%Y-%m-%d").dt.year().alias("Year"),
        pl.col("JobText").str.len_chars().alias("JobTextLength")
    ])

    year_range = major_data.select([
        pl.col("Year").min().alias("min"),
        pl.col("Year").max().alias("max")
    ]).row(0)

    print(f"\nTemporal coverage: {year_range[0]} - {year_range[1]}")

## Missing Skills Analysis

Before parsing skills, we check for missing `CanonSkillClusters` data. Jobs without skill clusters cannot be included in the skills analysis.

In [5]:
# Load data if not cached skills
if not skills_output_path.exists():
    # Check for missing skills
    total_before = len(major_data)
    major_data = major_data.filter(pl.col("CanonSkillClusters").is_not_null())
    total_after = len(major_data)
    missing_count = total_before - total_after

    print(f"\nMissing Skills Analysis:")
    print(f"  Records with skills: {total_after:,} ({total_after/total_before*100:.2f}%)")
    print(f"  Records without skills: {missing_count:,} ({missing_count/total_before*100:.2f}%)")

    if missing_count > 0:
        print(f"  Warning: {missing_count:,} records excluded from skills analysis")

## Skills Parsing: Wide to Long Format

We parse BGT skill clusters and transform the data from wide format (1 row per job) to long format (1 row per skill-job pair).

**BGT Skill Format:**
```
"Skill Family: Skill; Skill Type | Skill Family: Skill; Skill Type"
```

**Example:**
```
"Engineering: CAD;Technical Skills|Software: SolidWorks;Specialized Skills"
```

**Parsing robustness:**
- Handles complete entries (72.5%): Family: Skill; Type
- Handles missing type (0.0%): Uses "Specialized Skills" default
- Skips type-only entries (27.5%): "Specialized Skills" without family/skill
- Average result: 9.36 skills per job

**Expected output:** ~9M skill-job pairs from ~970k jobs

In [6]:
print("\n" + "=" * 80)
print("SKILLS PARSING AND TRANSFORMATION")
print("=" * 80)

if skills_output_path.exists():
    print(f"\nFound cached skills file: {skills_output_path}")
    print("Loading from cache (skipping transformation)...")
    skills_long_df = pl.read_parquet(skills_output_path)
    print(f"\nLoaded from cache")
else:
    print(f"\nNo cached file found at: {skills_output_path}")
    print("Running full transformation...")
    # Transform to long format using skills_utils
    skills_long_df = skills_utils.transform_to_long_format(
        major_data,
        id_col='JobID',
        skill_col='CanonSkillClusters',
        keep_cols=['Discipline', 'JobDate', 'Year', 'JobTextLength'],
        chunk_size=10000,
        verbose=True
    )
    print(f"\nTransformation complete")

print(f"  Skill-job pairs: {len(skills_long_df):,}")
print(f"  Unique jobs: {skills_long_df['JobID'].n_unique():,}")
print(f"  Unique skills: {skills_long_df['Skill'].n_unique():,}")


SKILLS PARSING AND TRANSFORMATION

Found cached skills file: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/skills_ME_ONLY.parquet
Loading from cache (skipping transformation)...

Loaded from cache
  Skill-job pairs: 9,356,249
  Unique jobs: 1,002,189
  Unique skills: 503


## Verify Columns

Job text length and year were calculated on `major_data` before the wide-to-long transformation. This is more memory efficient than joining full JobText to ~9M rows and then calculating length.

Job text length is correlated with number of skills (r ≈ 0.51). We use this as a covariate in our statistical models to control for this relationship.

In [7]:
# Verify columns are present (Year and JobTextLength were included in keep_cols)
print("Columns in skills_long_df:")
print(f"  {skills_long_df.columns}")
print(f"\nYear range: {skills_long_df['Year'].min()} - {skills_long_df['Year'].max()}")
print(f"JobTextLength range: {skills_long_df['JobTextLength'].min()} - {skills_long_df['JobTextLength'].max()}")

Columns in skills_long_df:
  ['JobID', 'Discipline', 'JobDate', 'Year', 'JobTextLength', 'SkillFamily', 'Skill', 'SkillType']

Year range: 2010 - 2022
JobTextLength range: 2 - 127283


## Export Long Format Skills

We save the long-format skills data for future use and as an intermediate checkpoint.

In [8]:
# Export skills_long_df (only if not loaded from cache)
if not skills_output_path.exists():
    skills_long_df.write_parquet(skills_output_path)
    print(f"\nLong format skills exported")
    print(f"  Path: {skills_output_path}")
    print(f"  Size: {skills_output_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print(f"\nSkipping export (using cached file)")
    print(f"  Path: {skills_output_path}")
    print(f"  Size: {skills_output_path.stat().st_size / 1024 / 1024:.2f} MB")


Skipping export (using cached file)
  Path: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/skills_ME_ONLY.parquet
  Size: 28.07 MB


## Skills Per Job Distribution

Summary statistics showing how many unique skills each job posting contains, along with a binned breakdown by skill count range.

In [9]:
# Compute and display skills per job distribution
distribution = skills_utils.compute_skills_per_job_distribution(skills_long_df)
skills_utils.display_skills_per_job_table(distribution)

Statistic,Value
Minimum,1.0 skill
Maximum,55.0 skills
Standard Deviation,4.46 skills
Mean,7.73 skills
Range,Jobs (% of total)
1-5 skills,"352,540 (35.18)"
6-10 skills,"411,148 (41.02)"
11-20 skills,"228,057 (22.76)"
21-50 skills,"10,434 (1.04)"
51+ skills,10 (0.00)


## Calculate Skill Prevalence

We calculate the proportion of jobs mentioning each skill in each year. This creates the time series data for temporal trend analysis.

**Prevalence formula:**
```
prevalence(skill, year) = (unique jobs with skill) / (total unique jobs in year)
```

**Note:** A job can mention the same skill multiple times (different families/types), but we count it only once per skill per year.

In [10]:
print("\n" + "=" * 80)
print("CALCULATING SKILL PREVALENCE")
print("=" * 80)

# Calculate prevalence by year
prevalence_df = skills_utils.calculate_skill_prevalence(
    skills_long_df,
    id_col='JobID',
    skill_col='Skill',
    year_col='Year',
    verbose=True
)

print(f"\nPrevalence calculation complete")
print(f"  Total records: {len(prevalence_df):,}")
print(f"  Unique skills: {prevalence_df['Skill'].n_unique():,}")
print(f"  Years: {prevalence_df['Year'].n_unique()}")


CALCULATING SKILL PREVALENCE
Calculating skill prevalence by year...
Calculated prevalence for 503 unique skills
Year range: 2010 - 2022
Total records: 6,231

Prevalence calculation complete
  Total records: 6,231
  Unique skills: 503
  Years: 13


## Add Job Text Length Covariate

We add standardized job text length to the prevalence data. This will be used as a covariate in all statistical models to control for the correlation between text length and skill count.

In [11]:
# Calculate mean job text length by year (using pre-calculated JobTextLength column)
job_text_lengths = skills_long_df.group_by("Year").agg(
    pl.col("JobTextLength").mean().alias("mean_length")
)


# Standardize
mean_overall = job_text_lengths['mean_length'].mean()
std_overall = job_text_lengths['mean_length'].std()

job_text_lengths = job_text_lengths.with_columns(
    ((pl.col("mean_length") - mean_overall) / std_overall).alias("job_text_length_std")
)

# Join with prevalence data
prevalence_df = prevalence_df.join(
    job_text_lengths.select(["Year", "job_text_length_std"]),
    on="Year",
    how="left"
)

print("\nAdded standardized job text length covariate")
print(f"  Overall mean job text length: {mean_overall:.2f}")
print(f"  Overall std dev job text length: {std_overall:.2f}")


Added standardized job text length covariate
  Overall mean job text length: 4689.94
  Overall std dev job text length: 583.22


## Filter to Common Skills

We filter to skills with ≥2% average prevalence across all years. This removes rare/niche skills that lack sufficient data for robust temporal trend analysis.

**Rationale for 2% threshold:**
- Ensures sufficient sample size per year
- Focuses on skills with meaningful workforce impact
- Removes one-off or highly specialized skills
- Standard threshold in workforce research

**Expected result:** ~200-250 common skills from thousands of unique skills

In [12]:
print("\n" + "=" * 80)
print("FILTERING TO COMMON SKILLS")
print("=" * 80)

# Filter using skills_utils
common_skills = skills_utils.filter_common_skills(
    prevalence_df,
    min_avg_prevalence=phase3_config['min_prevalence'],
    skill_col='Skill',
    verbose=True
)

# Filter prevalence data to common skills
prevalence_common = prevalence_df.filter(
    pl.col('Skill').is_in(common_skills)
)

print(f"\nFiltered to {len(common_skills)} common skills")
print(f"  Total prevalence records: {len(prevalence_common):,}")
print(f"  Average records per skill: {len(prevalence_common) / len(common_skills):.1f}")


FILTERING TO COMMON SKILLS
Filtering to skills with ≥2.0% average prevalence...
Skills meeting threshold: 77 / 503 (15.3%)
Highest average prevalence: 'Mechanical Engineering' (73.8%)

Filtered to 77 common skills
  Total prevalence records: 1,001
  Average records per skill: 13.0


# PART 2: Temporal Trend Analysis

## Statistical Modeling Approach

For each skill, we fit 5 candidate models and select the best using information-theoretic criteria:

**Models:**
1. **Null**: `prevalence ~ 1` (no temporal trend)
2. **Linear**: `prevalence ~ year + job_text_length_std`
3. **Log-year**: `prevalence ~ log(year) + job_text_length_std` (decelerating)
4. **Exponential**: `log(prevalence) ~ year + job_text_length_std`
5. **Quadratic**: `prevalence ~ year + year² + job_text_length_std` (accelerating)

**Model Selection:**
- Calculate AICc (corrected AIC for small samples, n=13)
- If overdispersion detected (c_hat > 1.0), use QAICc
- Calculate delta_i (difference from best AICc)
- Calculate Akaike weights
- Apply parsimony: if delta_i < 2, prefer simpler model

**Diagnostics:**
- Shapiro-Wilk: Test normality of residuals
- Breusch-Pagan: Test homoscedasticity
- Durbin-Watson: Test autocorrelation (acceptable: 1.5-2.5)

**Note:** With n=13 years, these tests have limited power. They serve as warning flags rather than definitive evidence.

In [13]:
print("\n" + "=" * 80)
print("PART 2: TEMPORAL TREND ANALYSIS")
print("=" * 80)

print(f"\nAnalyzing {len(common_skills)} skills...")
print(f"This will take ~30-60 minutes (1-3 seconds per skill)\n")


PART 2: TEMPORAL TREND ANALYSIS

Analyzing 77 skills...
This will take ~30-60 minutes (1-3 seconds per skill)



## Model Fitting Loop

We iterate through all common skills, fitting models and collecting results.

In [14]:
# Initialize results storage
skill_results = []
diagnostic_results = []
failed_skills = []

# Progress tracking
n_skills = len(common_skills)
progress_interval = max(1, n_skills // 10)  # Report every 10%

for idx, skill in enumerate(common_skills, 1):
    try:
        # Progress update
        if idx % progress_interval == 0 or idx == 1:
            print(f"Processing skill {idx}/{n_skills} ({idx/n_skills*100:.0f}%): {skill[:40]}...")
        
        # Fit models
        models = statistics_utils.fit_candidate_models(
            prevalence_common,
            skill,
            alpha=phase3_config['alpha']
        )
        
        # Select best model
        selection = statistics_utils.select_best_model(
            models,
            parsimony=True,
            delta_i_threshold=phase3_config['delta_i_threshold']
        )
        
        # Run diagnostics
        diagnostics = statistics_utils.run_diagnostics(
            models[selection.best_model],
            alpha=phase3_config['alpha']
        )
        
        # Assign two-dimensional evidence quality
        discriminability = statistics_utils.assign_discriminability(selection)
        assumption_quality = statistics_utils.assign_assumption_quality(selection, diagnostics)
        
        # Classify trajectory — indistinguishable models have no reportable winner
        if discriminability == 'Indistinguishable':
            trajectory = None
        else:
            trajectory = statistics_utils.classify_trajectory(
                selection,
                models[selection.best_model]
            )
        
        # Get start/end values
        skill_prev = prevalence_common.filter(pl.col('Skill') == skill).sort('Year')
        start_val = skill_prev['prevalence'].head(1).item()
        end_val = skill_prev['prevalence'].tail(1).item()

        # Get fitted values from best model for visualization
        best_model_result = models[selection.best_model]
        fitted_values = best_model_result.fitted_model.fittedvalues

        # Handle exponential model (needs back-transform from log space)
        if selection.best_model == 'exponential':
            fitted_values = np.exp(fitted_values)

        # Store result
        skill_results.append({
            'Skill': skill,
            'Selected_Model': selection.best_model,
            'Start_Value_2010': start_val,
            'End_Value_2022': end_val,
            'Absolute_Change': end_val - start_val,
            'Percent_Change': ((end_val - start_val) / start_val * 100) if start_val > 0 else 0,
            'Model_Coefficients': selection.model_coefficients,
            'Fitted_Values': fitted_values.tolist(),
            'R_squared': selection.r_squared,
            'Adj_R_squared': selection.adj_r_squared,
            'AICc': selection.aicc,
            'Delta_i': selection.delta_i,
            'Weight': selection.weight,
            'Discriminability': discriminability,
            'Assumption_Quality': assumption_quality,
            'Trajectory_Class': trajectory,
            'Mean_Prevalence': skill_prev['prevalence'].mean()
        })
        
        # Store diagnostics
        diagnostic_results.append({
            'Skill': skill,
            'Shapiro_W': diagnostics.shapiro_w,
            'Shapiro_p': diagnostics.shapiro_p,
            'Shapiro_Fail': diagnostics.shapiro_fail,
            'BP_stat': diagnostics.bp_stat,
            'BP_p': diagnostics.bp_p,
            'BP_Fail': diagnostics.bp_fail,
            'DW_stat': diagnostics.dw_stat,
            'DW_Fail': diagnostics.dw_fail,
            'Any_Fail': diagnostics.any_fail
        })
        
    except Exception as e:
        print(f"  Warning: Failed for skill: {skill} - {str(e)}")
        failed_skills.append(skill)

print(f"\nModel fitting complete")
print(f"  Successful: {len(skill_results)}/{n_skills}")
if failed_skills:
    print(f"  Failed: {len(failed_skills)} skills")

Processing skill 1/77 (1%): Mechanical Engineering...
Processing skill 7/77 (9%): System Design and Implementation...
Processing skill 14/77 (18%): Manufacturing Processes...
Processing skill 21/77 (27%): Civil and Architectural Engineering...
Processing skill 28/77 (36%): Packaging and Labeling...
Processing skill 35/77 (45%): Product Management...
Processing skill 42/77 (55%): Product Inspection...
Processing skill 49/77 (64%): Mathematical Software...
Processing skill 56/77 (73%): Schematic Diagrams...
Processing skill 63/77 (82%): Computer-Aided Manufacturing...
Processing skill 70/77 (91%): Supplier Relationship Management...
Processing skill 77/77 (100%): General Architecture...

Model fitting complete
  Successful: 77/77


## Create Summary DataFrames

We compile all results into structured DataFrames for export and visualization.

In [15]:
# Create summary DataFrame
summary_df = pl.DataFrame(skill_results)

# Create diagnostics DataFrame
diagnostics_df = pl.DataFrame(diagnostic_results)

print(f"\nCreated summary DataFrames")
print(f"  Summary records: {len(summary_df)}")
print(f"  Diagnostic records: {len(diagnostics_df)}")


Created summary DataFrames
  Summary records: 77
  Diagnostic records: 77


## Summary Statistics

Let's examine the distribution of results across model types, confidence tiers, and trajectory classifications.

In [16]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

# Model selection distribution
print("\nSelected Models:")
model_counts = summary_df.group_by('Selected_Model').agg(
    pl.count().alias('count')
).sort('count', descending=True)
for row in model_counts.iter_rows(named=True):
    pct = row['count'] / len(summary_df) * 100
    print(f"  {row['Selected_Model']}: {row['count']} ({pct:.1f}%)")

# Discriminability distribution
disc_order = ['Strong', 'Moderate', 'Weak', 'Indistinguishable']
print("\nModel Discriminability (Δᵢ to second-best):")
disc_counts = summary_df.group_by('Discriminability').agg(
    pl.count().alias('count')
)
disc_counts = disc_counts.with_columns(
    pl.col('Discriminability').cast(pl.Enum(disc_order))
).sort('Discriminability')
for row in disc_counts.iter_rows(named=True):
    pct = row['count'] / len(summary_df) * 100
    print(f"  {row['Discriminability']}: {row['count']} ({pct:.1f}%)")

# Assumption quality distribution
aq_order = ['Clean', 'Corrected', 'Compromised']
print("\nAssumption Quality:")
aq_counts = summary_df.group_by('Assumption_Quality').agg(
    pl.count().alias('count')
)
aq_counts = aq_counts.with_columns(
    pl.col('Assumption_Quality').cast(pl.Enum(aq_order))
).sort('Assumption_Quality')
for row in aq_counts.iter_rows(named=True):
    pct = row['count'] / len(summary_df) * 100
    print(f"  {row['Assumption_Quality']}: {row['count']} ({pct:.1f}%)")

# Trajectory distribution (excludes Indistinguishable skills where trajectory=None)
print("\nTrajectory Classifications (excluding Indistinguishable):")
traj_counts = summary_df.filter(
    pl.col('Trajectory_Class').is_not_null()
).group_by('Trajectory_Class').agg(
    pl.count().alias('count')
).sort('count', descending=True)
for row in traj_counts.iter_rows(named=True):
    pct = row['count'] / len(summary_df) * 100
    print(f"  {row['Trajectory_Class']}: {row['count']} ({pct:.1f}%)")


SUMMARY STATISTICS

Selected Models:
  null: 20 (26.0%)
  linear: 20 (26.0%)
  log_year: 18 (23.4%)
  exponential: 13 (16.9%)
  quadratic: 6 (7.8%)

Model Discriminability (Δᵢ to second-best):
  Strong: 41 (53.2%)
  Moderate: 32 (41.6%)
  Weak: 3 (3.9%)
  Indistinguishable: 1 (1.3%)

Assumption Quality:
  Clean: 48 (62.3%)
  Compromised: 29 (37.7%)

Trajectory Classifications (excluding Indistinguishable):
  Stable: 59 (76.6%)
  Rapidly Increasing: 7 (9.1%)
  Exponential Growth: 5 (6.5%)
  Accelerating: 3 (3.9%)
  Decelerating: 1 (1.3%)
  Non-monotonic: 1 (1.3%)


## Export Results

We export the summary and diagnostics to CSV files for publication and further analysis.

In [17]:
print("\n" + "=" * 80)
print("EXPORTING RESULTS")
print("=" * 80)

# Export summary - convert nested columns to JSON string for CSV compatibility
summary_path = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['trajectory_summary_file']

summary_export = summary_df.with_columns([
    pl.col('Model_Coefficients').map_elements(
        lambda x: json.dumps(x.to_list() if isinstance(x, pl.Series) else x) if x is not None else None,
        return_dtype=pl.Utf8
    ).alias('Model_Coefficients'),
    pl.col('Fitted_Values').map_elements(
        lambda x: json.dumps(x.to_list() if isinstance(x, pl.Series) else x) if x is not None else None,
        return_dtype=pl.Utf8
    ).alias('Fitted_Values')
])

summary_export.write_csv(summary_path)
print(f"\nExported skill trajectory summary")
print(f"  Path: {summary_path}")

# Export diagnostics
diagnostics_path = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['diagnostics_file']
diagnostics_df.write_csv(diagnostics_path)
print(f"\nExported regression diagnostics")
print(f"  Path: {diagnostics_path}")

# r3KMK6ys3SGQJpy


EXPORTING RESULTS

Exported skill trajectory summary
  Path: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/skill_trajectory_summary.csv

Exported regression diagnostics
  Path: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/regression_diagnostics.csv


## Generate IEEE-Format Publication Tables

We generate 8 IEEE-format tables for publication:
1. Top 20 Growing Skills
2. Top 20 Declining Skills
3. Stable High Prevalence Skills
4. High Demand Skills
5. Linear Trend Skills
6. Logarithmic Trend Skills
7. Quadratic Trend Skills
8. Exponential Trend Skills

In [18]:
print("\n" + "=" * 80)
print("GENERATING IEEE PUBLICATION TABLES")
print("=" * 80)

# Create output directory
tables_dir = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['ieee_tables_dir']
tables_dir.mkdir(parents=True, exist_ok=True)

# Generate all tables
publication_utils.export_all_ieee_tables(
    summary_df,
    str(tables_dir)
)

print(f"\nIEEE tables generated in: {tables_dir}")


GENERATING IEEE PUBLICATION TABLES
Exported Table_IEEE_1_Top_Growing_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_1_Top_Growing_Skills.csv
Exported Table_IEEE_2_Top_Declining_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_2_Top_Declining_Skills.csv
Exported Table_IEEE_3_Stable_High_Prevalence_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_3_Stable_High_Prevalence_Skills.csv
Exported Table_IEEE_4_High_Demand_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_4_High_Demand_Skills.csv
Exported Table_IEEE_5_Linear_Trend_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_5_Linear_Trend_Skills.csv
Exported Table_IEEE_6_Logarithmic_Trend_Skills to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/ieee_tables/Table_IEEE_6_Logarithmic_Trend_Skills.csv
Exported Table_IEEE_7_Quadratic_Trend_Skills to /Volumes/My Book/Burning

## Visualizations

Finally, we create trajectory visualizations organized by confidence tier.

In [19]:
print("\n" + "=" * 80)
print("GENERATING TRAJECTORY VISUALIZATIONS")
print("=" * 80)

# Create plots directory
plots_dir = Path(phase1_config['base_path']) / phase1_config['output_directory'] / phase3_config['plot_output_dir']
plots_dir.mkdir(parents=True, exist_ok=True)

# Generate trajectory grids by discriminability level
# Indistinguishable skills have no reportable trajectory and are excluded
for disc_level in ['Strong', 'Moderate', 'Weak']:
    output_path = plots_dir / f"{disc_level.lower()}_discriminability_trajectories.png"
    
    visualization_utils.plot_trajectory_grid(
        summary_df,
        prevalence_common,
        discriminability=disc_level,
        output_path=str(output_path),
        max_cols=5
    )

print(f"\nTrajectory plots saved to: {plots_dir}")


GENERATING TRAJECTORY VISUALIZATIONS
Saved Strong discriminability trajectories to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/plots/strong_discriminability_trajectories.png
Saved Moderate discriminability trajectories to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/plots/moderate_discriminability_trajectories.png
Saved Weak discriminability trajectories to /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/plots/weak_discriminability_trajectories.png

Trajectory plots saved to: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/plots


## Phase 3 Complete

### Summary

We successfully analyzed skill requirements and temporal trends in mechanical engineering jobs:

**Part 1: Skills Extraction**
1. Loaded ~982k mechanical-major jobs from Phase 2
2. Parsed ~9M skill-job pairs from BGT clusters
3. Calculated skill prevalence by year (2010-2022)
4. Filtered to ~200-250 common skills (>=2% prevalence)

**Part 2: Temporal Trend Analysis**
1. Fitted 5 candidate models per skill
2. Selected best models using AICc/QAICc with parsimony
3. Ran diagnostic tests (Shapiro-Wilk, BP, DW)
4. Assigned two-dimensional evidence quality (discriminability + assumption quality)
5. Classified trajectories (growth, decline, stable, etc.); Indistinguishable skills not reported
6. Generated 8 IEEE-format publication tables
7. Created trajectory visualizations by discriminability level

**Outputs:**
- `skills_ME_ONLY.parquet`: 9M skill-job pairs
- `skill_trajectory_summary.csv`: Complete analysis (includes `Discriminability`, `Assumption_Quality`)
- `regression_diagnostics.csv`: Test results
- 8 IEEE tables (CSV + combined LaTeX)
- Trajectory plots (Strong / Moderate / Weak discriminability)

### Key Findings Preview

**Distribution:**
- Skills analyzed: [see summary statistics]
- Strong discriminability: [see count]
- Growing skills: [see trajectory distribution]
- Declining skills: [see trajectory distribution]
- Stable skills: [see trajectory distribution]

### Evidence Quality Dimensions

Rather than collapsing evidence quality into a single tier, we report two independent dimensions per B&A (2002):

**Dimension 1 — Model Discriminability** (how clearly can we pick a winner?)

| Label | Δᵢ to second-best |
|---|---|
| Strong | ≤ 2 |
| Moderate | 2 < Δᵢ < 7 |
| Weak | 7 ≤ Δᵢ < 10 |
| Indistinguishable | ≥ 10 — trajectory not reported |

**Dimension 2 — Assumption Quality** (how much do we trust the AICc values?)

| Label | Condition |
|---|---|
| Clean | All diagnostics pass, no overdispersion |
| Corrected | Overdispersion detected; QAICc applied |
| Compromised | DW failure or ≥2 SW/BP failures |

This allows a reader to distinguish "Strong discriminability, compromised assumptions" from "Moderate discriminability, clean assumptions" — two meaningfully different evidence profiles that would previously both appear as Tier 2.

### Statistical Methodology Notes

**Why 5 models?**
- Captures different trajectory types (linear, accelerating, decelerating)
- Null model provides baseline for comparison
- Model selection prevents overfitting

**Why AICc instead of AIC?**
- Small sample size (n=13 years) requires correction
- AICc = AIC + (2K(K+1))/(n-K-1)
- Penalizes complexity more heavily with small n

**Why control for job text length?**
- Strong correlation (r~0.51) between length and skill count
- Text length has increased over time
- Including as covariate isolates temporal trends from data artifact

**Limitations:**
- Small sample (n=13) limits statistical power
- Diagnostic tests should be interpreted as warnings, not definitive
- Indistinguishable skills (Δᵢ ≥ 10) have no reportable trajectory

### References

- Alsharif, M. (2025). Engineering Employment Trends Study
- Burnham & Anderson (2002). Model Selection and Multimodel Inference
- AICc methodology for small samples
- Parsimony principle in model selection